# Forecasting with tabular foundational models

Tabular Foundation Models (FMs) are a new class of deep learning models that have been pre-trained on large amounts of tabular data so that they can be applied to a wide range of prediction tasks with minimal or no fine-tuning. These models have shown promising results in various domains, including forecasting. In this section, we will explore how to use tabular foundation models for forecasting tasks.



## Tabular Foundation Models vs. Machine Learning Models

Foundation models and traditional machine learning models approach forecasting in fundamentally different ways. Understanding these distinctions is crucial for knowing when and how to deploy each method.

**Zero-Shot Prediction**

Machine learning models require a training phase. You must fit the model on your historical target data (and exogenous variables) so the algorithm can learn the optimal weights and parameters for your specific time series. Foundation models are capable of zero-shot prediction. Because the model's parameters are already frozen from its massive pre-training phase, it can generate forecasts on your data immediately, without explicitly "learning" from your specific dataset first.

**The Role of the fit Method**

Calling .fit(y) in machine learning models triggers the optimization algorithm to minimize an error metric, updating the model's internal parameters. In foundation models, calling .fit(y) does not alter the model's neural network weights. Instead, the fit method acts as a standardizer. It is used merely to store necessary metadata (such as the last observed window of data, frequency, and scaling factors) so that the .predict() method has the exact context it needs to generate future values.

**Context Window vs. Engineered Lags**

Machine learning models rely on explicitly engineered features. In skforecast, you define lags or window_features to create a tabular dataset where past values are used as columns to predict the target. Foundation models rely on a context window. You pass a raw, sequential chunk of recent historical data (e.g., the last 512 observations) directly into the model at inference time. The attention mechanism inside the model automatically decides which past data points are most relevant.

## Tabular foundation models in skforecast

Foundational models that allow for regression tasks and follow the sklearn API (`fit` and `predict` methods) can be used with the `ForecasterRecursive`, `ForecasterRecursiveMultiSeries`, `ForecasterDirect` and `ForecasterDirectMultiVariate` classes. The following code snippets show how to use these models in skforecast.

<div class="admonition note" name="html-admonition" style="background: rgba(0,191,191,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #00bfa5; border-color: #00bfa5; padding-left: 10px; padding-right: 10px;">

<p class="title">
    <i style="font-size: 18px; color:#00bfa5;"></i>
    <b style="color: #00bfa5;">&#128161 Tip</b>
</p>

All these models can be run on a CPU; however, to unlock their full potential, it is highly recommended to use a GPU.
</div>

## Libraries and data

In [33]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent.parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
c:\Users\Joaquin\Documents\GitHub\skforecast
0.21.0


In [34]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from skforecast.recursive import ForecasterRecursive, ForecasterRecursiveMultiSeries
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_forecaster,
    backtesting_forecaster_multiseries
)
from skforecast.plot import set_dark_theme
from tabicl import TabICLRegressor
import torch
color = '\033[1m\033[38;5;208m' 
print(f"{color}torch version: {torch.__version__}")
print("  Cuda available :", torch.cuda.is_available())
print("  Device name    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch version: 2.6.0+cu124
  Cuda available : True
  Device name    : NVIDIA RTX 2000 Ada Generation Laptop GPU


In [35]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity')

# Aggregating in 1H intervals
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data.head(3)

╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 4 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

,Demand,Temperature,Holiday
Time,,,
2011-12-31 14:00:00,4323.095350,21.225,1.0
2011-12-31 15:00:00,3963.264688,20.625,1.0
2011-12-31 16:00:00,3950.913495,20.325,1.0


In [36]:
# Split data into train-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_test  = data.loc[end_train:, :].copy()

print(f"Train dates: {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Test dates : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")


Train dates: 2012-01-01 00:00:00 --- 2014-11-30 23:00:00  (n=25560)
Test dates : 2014-12-01 00:00:00 --- 2014-12-30 23:00:00  (n=720)


## TabICLv2

TabICLv2 is the premier open-source alternative, built for speed and high-volume data. It operates up to 10x faster than TabPFN and scales to datasets with 100,000+ rows using advanced KV-caching. Developed by Inria, it uses a "Fully Synthetic Pretraining" engine to ensure zero data leakage from real-world benchmarks. It is the go-to choice for commercial applications thanks to its permissive open-source license and native support for many-class classification and probabilistic regression.

In [37]:
# Create Forecaster
# ==============================================================================
# Activate KV Caching to speed up repeated inference
forecaster = ForecasterRecursive(
                 estimator = TabICLRegressor(kv_cache=True),
                 lags      = 15,
             )

In [ ]:
# Reduce the historic data to avoid huge context
data_train = data_train.iloc[-500:, :]

In [39]:
# Train Forecaster
# ==============================================================================
forecaster.fit(
    y=data_train["Demand"],
    exog=data_train[["Temperature", "Holiday"]],
    store_in_sample_residuals=True
)
forecaster

=================== 
ForecasterRecursive 
=================== 
Estimator: TabICLRegressor 
Lags: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] 
Window features: None 
Window size: 15 
Series name: Demand 
Exogenous included: True 
Exogenous names: Temperature, Holiday 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Training range: [Timestamp('2014-11-10 04:00:00'), Timestamp('2014-11-30 23:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Estimator parameters: 
    {'allow_auto_download': True, 'batch_size': 8, 'checkpoint_version': 'tabicl-
    regressor-v2-20260212.ckpt', 'device': None, 'disk_offload_dir': None,
    'feat_shuffle_method': 'latin', 'inference_config': None, 'kv_cache': True,
    'model_path': None, 'n_estimators': 8, 'n_jobs': None, 'norm_methods': None,
    'offload_mode': 'auto', 'outlier_threshold': 4.0, 'random_state': 42,
    'use_amp': 'auto', 'use_fa3': 'auto', 'verbose': False} 
fit_kwargs: {} 
Creation date: 2026-04-01 17:58:29 
Last fit date: 2026-04-01 17:58:31 
Skforecast version: 0.21.0 
Python version: 3.13.12 
Forecaster id: None

In [40]:
# Predictions: point forecast
# ==============================================================================
steps = 24
predictions = forecaster.predict(steps=steps, exog=data_test[["Temperature", "Holiday"]])
predictions.head(3)

2014-12-01 00:00:00    5633.830078
2014-12-01 01:00:00    5637.356934
2014-12-01 02:00:00    5611.678711
Freq: h, Name: pred, dtype: float64

In [41]:
# Predictions: intervals
# ==============================================================================
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    exog     = data_test[["Temperature", "Holiday"]],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head(3)

,pred,lower_bound,upper_bound
2014-12-01 00:00:00,5633.830078,5625.438525,5646.199707
2014-12-01 01:00:00,5637.356934,5623.425684,5660.406396
2014-12-01 02:00:00,5611.678711,5590.331885,5646.633105


In [42]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
        steps              = 24,
        initial_train_size = len(data.loc[:end_train]),
        refit              = False
)

metrics_levels, backtest_predictions = backtesting_forecaster(
    forecaster        = forecaster,
    y                 = data['Demand'],
    exog              = data[["Temperature", "Holiday"]],
    cv                = cv,
    metric            = 'mean_absolute_error',
    suppress_warnings = True
)

print("Backtest metrics")
display(metrics_levels)
print("")
print("Backtest predictions")
backtest_predictions.head(4)

KeyboardInterrupt: 

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, ax = plt.subplots(figsize=(7, 3))
data_test['Demand'].plot(ax=ax, label='test')
backtest_predictions['pred'].plot(ax=ax, label='predictions')
ax.legend();

<div class="admonition note" name="html-admonition" style="background: rgba(255,145,0,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #ff9100; border-color: #ff9100; padding-left: 10px; padding-right: 10px">

<p class="title">
    <i style="font-size: 18px; color:#ff9100; border-color: #ff1744;"></i>
    <b style="color: #ff9100;"> <span style="color: #ff9100;">&#9888;</span> Warning</b>
</p>

These results are for illustrative purposes. Since this is a widely available public dataset, it is highly probable that the model was exposed to these specific data points during its pre-training phase. As a result, the predictions may be more optimistic than what would be achieved in a real-world production environment.

</div>

With the class `ForecasterRecursiveMultiSeries` it is possible to model multiple time series at the same time.

In [ ]:
# Data
# ==============================================================================
data_multiseries = fetch_dataset(name="items_sales")
display(data_multiseries.head(3))

In [ ]:
# Split data into train-test
# ==============================================================================
end_train = '2014-07-15 23:59:00'
data_multiseries_train = data_multiseries.loc[:end_train, :]
data_multiseries_test  = data_multiseries.loc[end_train:, :]

In [ ]:
# Create and train Forecaster
# ==============================================================================
forecaster = ForecasterRecursiveMultiSeries(
                 estimator = TabICLRegressor(),
                 lags      = 15,
             )
forecaster.fit(series=data_multiseries_train)
forecaster

In [ ]:
# Predictions and prediction intervals
# ==============================================================================
steps = 24

# Predictions for item_1 and item_2
predictions_item_1 = forecaster.predict(steps=steps, levels=['item_1', 'item_2'])
display(predictions_item_1.head())
print("")

# Interval predictions for item_1 and item_2
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    levels   = ['item_1', 'item_2'],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head()

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 5), sharex=True)

predictions = forecaster.predict(steps=len(data_multiseries_test), levels=data_multiseries.columns)
for i, col in enumerate(data_multiseries.columns):
    data_multiseries_train[col].plot(ax=axes[i], label='train')
    data_multiseries_test[col].plot(ax=axes[i], label='test')
    predictions.query(f"level == '{col}'").plot(ax=axes[i], label='predictions', color='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('sales')
    axes[i].set_xlabel('')
    axes[i].legend(loc='upper left')

fig.tight_layout()
plt.show();

## TabPFN-2.5

TabPFN-2.5 is the current benchmark for zero-shot tabular learning. Pre-trained on millions of synthetic datasets, it handles up to 50,000 samples and 2,000 features in a single forward pass without task-specific training. Its standout feature is a distillation engine that can compress the transformer into a lightweight, production-ready MLP or tree ensemble. While highly accurate and scikit-learn compliant, its weights are gated under a non-commercial license, requiring an API subscription for enterprise use.

Because TabPFN-2.5 is licensed under a custom Non-Commercial License, the weights are "gated" on Hugging Face. To download them:

1) Visit the Repository: Go to Prior-Labs/tabpfn_2_5 (or the latest version like tabpfn_2_6 if preferred).

2) Accept Terms: You must be logged into Hugging Face and click the button to agree to the license terms and share your contact information with Prior Labs.

2) Authentication: When you run the code for the first time, the package will prompt you for a Hugging Face Token to verify your access. You can generate this in your Hugging Face account settings under "Access Tokens."

In [ ]:
# Create Forecaster
# ==============================================================================
# Activate KV Caching to speed up repeated inference
forecaster = ForecasterRecursive(
                 estimator = TabICLRegressor(fit_mode='fit_with_cache'),
                 lags      = 15,
             )

Checkpoint 'tabicl-regressor-v2-20260212.ckpt' not cached.



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f9264630-af13-4fc4-b8ab-4b5ba79b1f54)')' thrown while requesting HEAD https://huggingface.co/jingang/TabICL/resolve/main/tabicl-regressor-v2-20260212.ckpt
Retrying in 1s [Retry 1/5].
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tabicl-regressor-v2-20260212.ckpt:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/jingang/TabICL/resolve/main/tabicl-regressor-v2-20260212.ckpt: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


tabicl-regressor-v2-20260212.ckpt:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/jingang/TabICL/resolve/main/tabicl-regressor-v2-20260212.ckpt: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e09a7628-9783-4ecb-b4df-60446c7a3830)')' thrown while requesting GET https://huggingface.co/jingang/TabICL/resolve/main/tabicl-regressor-v2-20260212.ckpt
Retrying in 1s [Retry 1/5].


tabicl-regressor-v2-20260212.ckpt:   0%|          | 0.00/114M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/jingang/TabICL/resolve/main/tabicl-regressor-v2-20260212.ckpt: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


In [ ]:
# Train Forecaster
# ==============================================================================
forecaster.fit(
    y=data_train["Demand"],
    exog=data_train[["Temperature", "Holiday"]],
    store_in_sample_residuals=True
)
forecaster

In [ ]:
# Predictions: point forecast
# ==============================================================================
steps = 24
predictions = forecaster.predict(steps=steps, exog=data_test[["Temperature", "Holiday"]])
predictions.head(3)

In [ ]:
# Predictions: intervals
# ==============================================================================
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    exog     = data_test[["Temperature", "Holiday"]],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head(3)

In [ ]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
        steps              = 24,
        initial_train_size = len(data.loc[:end_train]),
        refit              = False
)

metrics_levels, backtest_predictions = backtesting_forecaster(
    forecaster        = forecaster,
    series            = data['Demand'],
    exog              = data[["Temperature", "Holiday"]],
    cv                = cv,
    metric            = 'mean_absolute_error',
    suppress_warnings = True
)

print("Backtest metrics")
display(metrics_levels)
print("")
print("Backtest predictions")
backtest_predictions.head(4)

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, ax = plt.subplots(figsize=(7, 3))
data_test['Demand'].plot(ax=ax, label='test')
backtest_predictions['pred'].plot(ax=ax, label='predictions')
ax.legend();

<div class="admonition note" name="html-admonition" style="background: rgba(255,145,0,.1); padding-top: 0px; padding-bottom: 6px; border-radius: 8px; border-left: 8px solid #ff9100; border-color: #ff9100; padding-left: 10px; padding-right: 10px">

<p class="title">
    <i style="font-size: 18px; color:#ff9100; border-color: #ff1744;"></i>
    <b style="color: #ff9100;"> <span style="color: #ff9100;">&#9888;</span> Warning</b>
</p>

These results are for illustrative purposes. Since this is a widely available public dataset, it is highly probable that the model was exposed to these specific data points during its pre-training phase. As a result, the predictions may be more optimistic than what would be achieved in a real-world production environment.

</div>

With the class `ForecasterRecursiveMultiSeries` it is possible to model multiple time series at the same time.

In [ ]:
# Data
# ==============================================================================
data_multiseries = fetch_dataset(name="items_sales")
display(data_multiseries.head(3))

In [ ]:
# Split data into train-test
# ==============================================================================
end_train = '2014-07-15 23:59:00'
data_multiseries_train = data_multiseries.loc[:end_train, :]
data_multiseries_test  = data_multiseries.loc[end_train:, :]

In [ ]:
# Create and train Forecaster
# ==============================================================================
forecaster = ForecasterRecursiveMultiSeries(
                 estimator = TabICLRegressor(),
                 lags      = 15,
             )
forecaster.fit(series=data_multiseries_train)
forecaster

In [ ]:
# Predictions and prediction intervals
# ==============================================================================
steps = 24

# Predictions for item_1 and item_2
predictions_item_1 = forecaster.predict(steps=steps, levels=['item_1', 'item_2'])
display(predictions_item_1.head())
print("")

# Interval predictions for item_1 and item_2
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    levels   = ['item_1', 'item_2'],
    interval = [10, 90],  # 80% prediction interval
)
predictions_intervals.head()

In [ ]:
# Plot predictions
# ==============================================================================
set_dark_theme()
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 5), sharex=True)

predictions = forecaster.predict(steps=len(data_multiseries_test), levels=data_multiseries.columns)
for i, col in enumerate(data_multiseries.columns):
    data_multiseries_train[col].plot(ax=axes[i], label='train')
    data_multiseries_test[col].plot(ax=axes[i], label='test')
    predictions.query(f"level == '{col}'").plot(ax=axes[i], label='predictions', color='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('sales')
    axes[i].set_xlabel('')
    axes[i].legend(loc='upper left')

fig.tight_layout()
plt.show();

## Speed up inference with Self-Distillation

Model Distillation (also known as Knowledge Distillation) is a machine learning technique where a small, efficient model (the Student) is trained to mimic the behavior and performance of a large, complex model (the Teacher).

Think of it as a mentor-apprentice relationship: the Teacher has the "wisdom" (complex patterns and high accuracy), but it is too slow or "heavy" for practical use. The Student is leaner and faster, learning to reach the same conclusions as the Teacher without needing the same massive computational resources.

As we discussed regarding TabPFN-2.5, the primary bottleneck of foundational tabular models is their transformer-based architecture, which can be slow during inference.

Distillation solves this by converting the foundation model into a compact MLP or tree ensemble. This allows you to:

Reduce Latency: Get predictions in milliseconds instead of seconds.

Lower Costs: Run the model on cheaper hardware (CPU instead of high-end GPU).

Deploy Anywhere: Use the "smarts" of a massive foundation model in environments with strict memory limits, like edge devices or real-time web services.

In [ ]:
from tabicl import TabICLRegressor
from lightgbm import LGBMRegressor

In [ ]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity')

# Aggregating in 1H intervals
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data.head(3)

# Split data into train-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_test  = data.loc[end_train:, :].copy()

╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 4 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

In [ ]:
# 1. INITIALIZE TEACHER (Foundational Model)
# We enable kv_cache=True to speed up the subsequent prediction pass on X_train.
teacher = TabICLRegressor(kv_cache=True, ) 

# 2. "FIT" TEACHER
# In TabICL, this stores X_train and y_train as the context for In-Context Learning.
forecaster = ForecasterRecursive(
                 estimator = LGBMRegressor(),
                 lags      = 15,
             )
X_train, y_train = forecaster.create_train_X_y(
    y=data_train["Demand"],
    exog=data_train[["Temperature", "Holiday"]]
)
teacher.fit(X_train, y_train)

# 3. GENERATE SOFT TARGETS (Self-Distillation)
# We use the Teacher to predict values for the same X_train it was "fitted" on.
# These "soft targets" represent the Teacher's refined interpretation of the data.
soft_targets = teacher.predict(X_train)

# 4. TRAIN THE STUDENT (Fast Model)
# The Student (XGBoost) learns to map X_train to the Teacher's soft_targets.
student = LGBMRegressor(
    max_depth=6
)
student.fit(X_train, soft_targets)

# 5. Create Forecaster
# The student is now a "distilled" estimator that can be used in skforecast.
forecaster.estimator = student

KeyboardInterrupt: 

In [ ]:
# Predictions: point forecast
# ==============================================================================
steps = 24
predictions = forecaster.predict(steps=steps, exog=data_test[["Temperature", "Holiday"]])
predictions.head(3)